# Coop Case — Q4.2: Do Discounts Drive More Frequent Visits — and Can We Use That for Cross-Channel Growth?

**Question:** Does giving a customer a discount make them come to the store more often? If so, how
does that affect sales, and can we use it to grow omni-channel adoption?

Sub-questions:
- Is discount usage correlated with visit frequency?
- Does that hold up once we control for the obvious confound (frequent shoppers naturally rack up
  more discounted baskets just by shopping more)?
- Does discount usage relate to becoming an omni-channel customer?


## 1. Setup & load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)


In [ ]:
DATA_PATH = "2months_v2/rl_2months.csv"

dtypes = {
    "receiptKey": "int64",
    "hourOfDay": "int8",
    "minuteOfHour": "int8",
    "quantity": "float32",
    "lineItemAmount": "float32",
    "lineItemAmountExclVat": "float32",
    "discountAmountExclVat": "float32",
    "lineItemCostExclVat": "float32",
    "CoopOnlineYN": "category",
    "store": "category",
    "customerId": "Int64",
    "householdId": "Int64",
    "MOSAICGroup": "category",
    "MOSAICGroupDescription": "category",
    "MOSAICType": "category",
    "MOSAICTypeDescription": "category",
    "DominantBuyingPowerClass": "category",
    "ItemID": "int64",
    "ItemSubSegmentName": "category",
    "ItemSubSegmentID": "Int64",
    "ItemSegmentName": "category",
    "ItemSegmentID": "Int64",
    "ItemSubCategoryName": "category",
    "ItemSubCategoryID": "Int64",
    "ItemCategoryName": "category",
    "ItemCategoryID": "Int64",
    "ItemCategoryTeamName": "category",
    "ItemCategoryTeamID": "Int64",
    "ItemCategoryGroupName": "category",
    "ItemCategoryGroupID": "Int64",
    "ItemCategoryAreaName": "category",
    "ItemCategoryAreaID": "Int64",
    "Brand": "category",
    # Stored as floats in the source file (0.0 / 1.0), not clean ints -- pandas
    # won't safely downcast float64 -> int8 during read_csv, so keep as float32.
    "eko": "float32",
    "organic": "float32",
    "krav": "float32",
    "fair_trade": "float32",
    "msc": "float32",
    "no_lactose": "float32",
}

df = pd.read_csv(
    DATA_PATH,
    dtype=dtypes,
    parse_dates=["DayDate"],
)
df["profit"] = df["lineItemAmountExclVat"] - df["lineItemCostExclVat"]

print(df.shape)
df.head()

## 2. Build basket- and household-level discount metrics

A basket counts as "discounted" if its total `discountAmountExclVat` is negative (any discount
applied). `discount_rate` = share of a household's baskets that included a discount.


In [ ]:
df["channel"] = df["CoopOnlineYN"].map({"Y": "Online", "N": "Offline"}).astype(str)

basket = df.groupby("receiptKey", observed=True).agg(
    householdId=("householdId", "first"),
    channel=("channel", "first"),
    basket_value=("lineItemAmountExclVat", "sum"),
    discount=("discountAmountExclVat", "sum"),
).reset_index()
basket["discounted"] = basket["discount"] < 0

household = basket.groupby("householdId", observed=True).agg(
    n_baskets=("receiptKey", "count"),
    total_spend=("basket_value", "sum"),
    discounted_baskets=("discounted", "sum"),
).reset_index()
household["discount_rate"] = household["discounted_baskets"] / household["n_baskets"]
household["avg_spend_per_basket"] = household["total_spend"] / household["n_baskets"]

print(household.shape)
household["discount_rate"].describe()


## 3. Raw correlation: discount usage vs. visit frequency

The naive question first -- does discount_rate simply correlate with how often a household shops?


In [ ]:
corr_freq = household["discount_rate"].corr(household["n_baskets"])
corr_spend = household["discount_rate"].corr(household["avg_spend_per_basket"])
print(f"Correlation: discount_rate vs. n_baskets        = {corr_freq:.3f}")
print(f"Correlation: discount_rate vs. avg_spend_per_basket = {corr_spend:.3f}")


In [ ]:
def bucket(r):
    if r == 0:
        return "Never discounted"
    if r < 0.5:
        return "Sometimes (<50%)"
    return "Often (>=50%)"

household["discount_group"] = household["discount_rate"].apply(bucket)

group_summary = household.groupby("discount_group", observed=True).agg(
    n_households=("householdId", "count"),
    avg_baskets=("n_baskets", "mean"),
    avg_spend_per_basket=("avg_spend_per_basket", "mean"),
).round(2)
group_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
order = ["Never discounted", "Sometimes (<50%)", "Often (>=50%)"]
group_summary.loc[order, "avg_baskets"].plot(kind="bar", ax=axes[0], color="#22B573")
axes[0].set_title("Avg baskets (visit frequency) by discount usage group")
axes[0].set_ylabel("Avg baskets per household")
axes[0].tick_params(axis="x", rotation=20)

group_summary.loc[order, "avg_spend_per_basket"].plot(kind="bar", ax=axes[1], color="#8FA097")
axes[1].set_title("Avg spend per basket by discount usage group")
axes[1].set_ylabel("SEK")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


## 4. Controlling for the frequency confound

A household that shops 20 times has 20 chances to hit a discounted basket; a household that shops
once has one chance. So "discount_rate correlates with spend per basket" could just mean "bigger
baskets are more likely to contain *something* on promotion" -- not that discounts *cause* bigger
or more frequent shopping. Restrict to households within a **similar visit-frequency band** (3-8
baskets) before comparing groups, the same technique used for the buying-power confound in Q1.


In [ ]:
controlled = household[(household["n_baskets"] >= 3) & (household["n_baskets"] <= 8)]
print(f"Households in controlled band (3-8 baskets): {controlled.shape[0]:,}\n")

controlled.groupby("discount_group", observed=True).agg(
    n=("householdId", "count"),
    avg_baskets=("n_baskets", "mean"),
    avg_spend_per_basket=("avg_spend_per_basket", "mean"),
).round(2)


## 5. Discount usage vs. omni-channel status

In [ ]:
household_channels = basket.groupby("householdId", observed=True)["channel"].agg(lambda s: set(s))
household_mix = household_channels.apply(
    lambda s: "Omni-channel" if len(s) > 1 else "Single-channel"
)
household["mix"] = household["householdId"].map(household_mix)

print("Omni-channel share by discount group -- UNCONTROLLED (whole population):")
print((household.groupby("discount_group", observed=True)["mix"]
       .apply(lambda s: (s == "Omni-channel").mean() * 100)).round(2))
print()

controlled = controlled.copy()
controlled["mix"] = controlled["householdId"].map(household_mix)
print("Omni-channel share by discount group -- CONTROLLED (3-8 basket households only):")
print((controlled.groupby("discount_group", observed=True)["mix"]
       .apply(lambda s: (s == "Omni-channel").mean() * 100)).round(2))


## 6. Takeaways

- **Raw correlation is weak/near-zero for frequency, moderate for basket size**: discount_rate vs.
  n_baskets correlation ≈ **-0.05** (essentially no relationship — more discounts does NOT mean more
  visits, if anything a tiny negative drift). discount_rate vs. avg_spend_per_basket ≈ **+0.34**
  (moderate positive) — but this likely reflects reverse causation: bigger baskets are simply more
  likely to contain a promo-eligible item, not "discounts cause bigger baskets."

- **The 3-way group breakdown is non-monotonic, not a clean story**: "Never discounted" households
  average only 1.35 baskets (148.99 SEK/basket — one-off, small shoppers); "Often discounted"
  (≥50% of baskets) average 4.45 baskets at the highest spend/basket (419.39 SEK); but "Sometimes"
  discounted (<50%) average **10.80 baskets** — the most frequent shoppers of all, at a lower spend/basket
  (173.73 SEK). In other words, the group that ISN'T majority-discounted is actually the most
  frequent-visit group. This alone kills a simple "more discounts -> more visits" narrative.

- **The apparent discount->omni-channel link is almost entirely a frequency confound.** Uncontrolled,
  "Sometimes"-discounted households show a startling 41% omni-channel rate vs. 0% for "Never" — but
  after restricting to a matched visit-frequency band (3-8 baskets), that gap collapses to
  **0.33% ("Sometimes") vs. 1.18% ("Often") vs. 0% ("Never")** — all small, close together, and on
  tiny sample sizes. The uncontrolled number was almost entirely explained by "more visits = more
  chances to have tried both channels," not a real discount effect.

- **Bottom line: discounts do not look like a reliable lever for driving repeat visits or
  cross-channel adoption in this data.** The relationship that looked promising in the raw numbers
  mostly evaporates once frequency is controlled for. This is a *correlational* dataset with no
  natural experiment (discount timing/eligibility isn't randomized), so this can't fully rule out a
  real effect — but it means Coop shouldn't assume "give more discounts -> more visits/omni
  adoption" without testing it directly (e.g. an actual A/B discount-targeting pilot with a held-out
  control group), rather than inferring it from this observational 2-month window.

- **What would actually test this properly**: a controlled pilot — offer a subset of single-channel
  customers a discount specifically redeemable on the *other* channel, and measure conversion against
  a held-out control group. That's the only way to establish causation here.
